# Re-run: robustness (RQ2)

This notebook recounts RQ2's headline numbers from the benchmark repository's committed outputs:
`out/leaderboard.json` (each probe's effect with its interval, per model and task), `out/overall.json`
(the board's flags), `paraphrase/out/paraphrase.json` (the rewording study) and `hosting/out/hosting.json`
(8-bit against 4-bit hosts). Flags are recounted from the effects and the board's 5% rule, not only read.

In [1]:
# Finding the published data. This cell is the same in every notebook of this report.
import gzip, json, os, sys
from pathlib import Path

# Importing from the release's code/ folder must not leave __pycache__ folders in it.
sys.dont_write_bytecode = True

RELEASE_ID = "ai-grading-benchmark-2026"   # set when the report is built
DATA_NAME = "per-case-model-outputs.jsonl"


def find_release_file(name=DATA_NAME):
    """Look beside this notebook, in RELEASE_DIR, in the folders above, and in a repository checkout."""
    here = Path.cwd()
    places = [Path(os.environ["RELEASE_DIR"])] if os.environ.get("RELEASE_DIR") else []
    places += [here, *here.parents]
    places += [p / "research/wiki/research/releases" / RELEASE_ID for p in [here, *here.parents]]
    for d in places:
        for n in (name, name + ".gz"):
            if (d / n).is_file():
                return d / n
    raise FileNotFoundError(
        f"Could not find {name} (or {name}.gz). Put it in the same folder as this notebook: download it "
        f"from the report ('Download the data file', beside each notebook) or take it from the release "
        f"directory. Or set the environment variable RELEASE_DIR to the folder that holds it.")


def per_case(runs=None):
    """Yield the per-case records of the published file, optionally only for the named runs."""
    path = find_release_file()
    opener = gzip.open if path.suffix == ".gz" else open
    with opener(path, "rt", encoding="utf-8") as f:
        for line in f:
            r = json.loads(line)
            if r.get("record") == "case" and (runs is None or r["run"] in runs):
                yield r


def code_dir():
    """The release's code package (its code/ folder): in RELEASE_DIR, beside this notebook or in a folder
    above it (as when the notebook sits in the unzipped release's notebooks/ folder), in a folder named
    after the release there, or in a repository checkout."""
    here = Path.cwd()
    places = [Path(os.environ["RELEASE_DIR"])] if os.environ.get("RELEASE_DIR") else []
    for p in [here, *here.parents]:
        places += [p, p / RELEASE_ID, p / "research/wiki/research/releases" / RELEASE_ID]
    for d in places:
        if (d / "code" / "SHA256SUMS").is_file():
            return d / "code"
    raise FileNotFoundError(
        f"Could not find the code package of release {RELEASE_ID} (its code/ folder). Download the release "
        f"from the report ('Download the release', in the sidebar or beside each notebook), unzip it, and "
        f"run this notebook from the unzipped folder or its notebooks/ folder. Or set the environment "
        f"variable RELEASE_DIR to the unzipped {RELEASE_ID} folder.")


def find_code_file(relpath):
    """A file inside the release's code package, by its path there (e.g. studies/.../out/result.json)."""
    path = code_dir() / relpath
    if not path.is_file():
        raise FileNotFoundError(f"The code package at {code_dir()} has no {relpath}: is it release {RELEASE_ID}?")
    return path


def load_code_json(relpath):
    """A committed output of the code package, parsed."""
    return json.loads(find_code_file(relpath).read_text(encoding="utf-8"))


def shown(path):
    """A path as printed: relative to this notebook's folder, so an executed notebook names no machine's folders."""
    return os.path.relpath(path, Path.cwd())


def corpus_file(env_var, name):
    """A public corpus file the reader has downloaded: $env_var/name, or name beside this notebook."""
    for d in ([Path(os.environ[env_var])] if os.environ.get(env_var) else []) + [Path.cwd()]:
        if (d / name).is_file():
            return d / name
    raise FileNotFoundError(f"This notebook also needs {name} from the public corpus. Download the corpus "
                            f"(its location and hash are in the release file), then set {env_var} to its "
                            f"folder or put {name} beside this notebook.")


SOURCE_ENV, SOURCE_NAME = "GBENCH_DIR", "ai-grading-benchmark"   # set when a release's code is another repository


def source_repo():
    """The repository a release's code and committed outputs are published in (a release whose paper cites
    it rather than a copied code package): $SOURCE_ENV, or a checkout named SOURCE_NAME beside this
    notebook or beside a folder above it."""
    here = Path.cwd()
    places = [Path(os.environ[SOURCE_ENV])] if SOURCE_ENV and os.environ.get(SOURCE_ENV) else []
    places += [p / SOURCE_NAME for p in [here, *here.parents]]
    for d in places:
        if d.is_dir() and (d / "out").is_dir():
            return d
    raise FileNotFoundError(
        f"Could not find the {SOURCE_NAME} repository this release's code is published in. Clone it at the "
        f"commit the release file names, then set the environment variable {SOURCE_ENV} to the clone, or "
        f"run this notebook from a folder beside it.")


def load_source_json(relpath):
    """A committed output of that repository, by its path there (cited in the paper as code/<path>)."""
    return json.loads((source_repo() / relpath).read_text(encoding="utf-8"))


print("Source repository:", shown(source_repo()))

Source repository: ../../../../../../ai-grading-benchmark


In [2]:
from decimal import Decimal, ROUND_HALF_UP

def p(x, d=1, sign=False):
    v = Decimal(repr(float(x))).quantize(Decimal(1).scaleb(-d), rounding=ROUND_HALF_UP)
    return f"{v:+.{d}f}" if sign else f"{v:.{d}f}"

lb = [r for r in load_source_json("out/leaderboard.json")["rows"] if r["task"] in ("mohler", "code", "asap2")]
models = sorted({r["model"] for r in lb})
gamed = {m for m in models if any(r["gaming_max_share"]["estimate"] > 0.05 for r in lb if r["model"] == m)}
blank = {m for m in models if any(r["blank_credited"] > 0 for r in lb if r["model"] == m)}
flags = {r["model"]: r["flags"] for r in load_source_json("out/overall.json")["rows"] if r.get("rank")}
print(len(models), "models;", len(gamed), "gamed by more than 5% of the scale;", len(blank), "credit blank answers")
assert (len(models), len(gamed), len(blank & gamed)) == (22, 14, 7) and blank <= gamed
assert {m for m, f in flags.items() if f} == gamed      # the board's flags are exactly these
top = max(lb, key=lambda r: r["gaming_max_share"]["estimate"])
print("largest gain:", top["model"], top["task"], p(100 * top["gaming_max_share"]["estimate"], sign=True))
assert (top["model"], top["task"], p(100 * top["gaming_max_share"]["estimate"], sign=True)) == ("qwen/qwen3-235b-a22b-2507", "code", "+83.2")

22 models; 14 gamed by more than 5% of the scale; 7 credit blank answers
largest gain: qwen/qwen3-235b-a22b-2507 code +83.2


**Strong against weak student.** The same answer described as from a strong or a weak student: pairs
whose interval lies above 0 (the label raised the grade) or below 0.

In [3]:
raised = sum(r["attainment_label_share"]["ci95"][0] > 0 for r in lb)
lowered = sum(r["attainment_label_share"]["ci95"][1] < 0 for r in lb)
print("label raised the grade in", raised, "of", len(lb), "pairs; lowered it in", lowered)
assert (raised, lowered, len(lb)) == (28, 0, 66)

label raised the grade in 28 of 66 pairs; lowered it in 0


In [4]:
pa = load_source_json("paraphrase/out/paraphrase.json")["models"]
sensitive = sum(v["verdict"] == "PROMPT-SENSITIVE" and v["excess"]["ci95"][0] > 0 for v in pa.values())
print("PROMPT-SENSITIVE:", sensitive, "of", len(pa))
assert (sensitive, len(pa)) == (11, 22)
ho = load_source_json("hosting/out/hosting.json")["comparisons"]
assert all(c["verdict"] == "PRECISION MATTERS" and c["forged_gain_fp8_minus_fp4"]["ci95"][1] < 0 for c in ho) and len(ho) == 3
print("all three hosting comparisons: the forged override gains more at 4-bit (8-bit minus 4-bit below 0)")

PROMPT-SENSITIVE: 11 of 22
all three hosting comparisons: the forged override gains more at 4-bit (8-bit minus 4-bit below 0)


**Guard sentence against re-ask noise** (`guard-retest/out/guard-retest.json`), **precision with the host held
fixed** (`precision-controlled/out/precision-controlled.json`) and **the ranking across prompts**
(`prompt-ranking/out/ranking.json`).

In [5]:
gr = load_source_json("guard-retest/out/guard-retest.json")["models"]
more = sorted(m for m, v in gr.items() if v["verdict"] == "GUARD MAKES IT MORE GAMEABLE")
print("guard makes it more gameable:", more)
assert more == ["allenai/olmo-3.1-32b-instruct", "google/gemma-3-27b-it", "google/gemma-3-4b-it"]
assert gr["qwen/qwen3-235b-a22b-2507"]["verdict"] == "GUARD MAKES IT LESS GAMEABLE"
assert round(100 * gr["allenai/olmo-3.1-32b-instruct"]["effect_G"]["estimate"], 1) == 9.2
pc = load_source_json("precision-controlled/out/precision-controlled.json")
print("controlled precision:", pc["overall"])
assert pc["overall"] == "PRECISION ALONE NOT SHOWN TO RAISE GAMING"
rk = load_source_json("prompt-ranking/out/ranking.json")
print("ranking across prompts:", rk["verdict"], p(rk["D"]["estimate"], 3, sign=True))
assert rk["verdict"] == "RANKING WITHIN RE-ASK NOISE" and p(rk["D"]["estimate"], 3) == "-0.019"

guard makes it more gameable: ['allenai/olmo-3.1-32b-instruct', 'google/gemma-3-27b-it', 'google/gemma-3-4b-it']
controlled precision: PRECISION ALONE NOT SHOWN TO RAISE GAMING
ranking across prompts: RANKING WITHIN RE-ASK NOISE -0.019
